In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.optimize import minimize_scalar
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from venn_abers import VennAbersCalibrator
    HAS_VENN_ABERS = True
except Exception:
    HAS_VENN_ABERS = False


RANDOM_STATE = 42
DATA_PATH = "/kaggle/input/datasets/umerrtx/machine-failure-prediction-using-sensor-data/data.csv"
TARGET = "fail"
TEST_SIZE = 0.15
VALIDATION_SHARE_OF_REMAINDER = 0.17647058823529413

ROW_CHECK_COST = 10.0
BASE_MISSED_LOSS = 500.0
RECOVERY_RATE = 1.0
BOOTSTRAP_ROUNDS = 80
N_SPLITS = 4

SEVERITY_SHRINK_ALPHA = 0.50
CAPACITY_VALUE_RETENTION = 0.95
WEIGHT_STYLES = ["none", "balanced_x0.75", "balanced", "balanced_x1.25", "balanced_x1.5"]


def print_header(title: str):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def clip_probs(p):
    return np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)


def calibration_stats(y_true, scores, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    scores = clip_probs(scores)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(scores, bins[1:-1], right=True)

    rows = []
    ece = 0.0
    max_gap = 0.0
    n = len(scores)

    for b in range(n_bins):
        mask = bin_ids == b
        count = int(mask.sum())
        if count == 0:
            continue

        avg_pred = float(scores[mask].mean())
        avg_true = float(y_true[mask].mean())
        gap = abs(avg_true - avg_pred)

        rows.append({
            "bin": b,
            "count": count,
            "mean_pred": avg_pred,
            "observed_rate": avg_true,
            "gap": gap,
        })

        ece += (count / n) * gap
        max_gap = max(max_gap, gap)

    return pd.DataFrame(rows), float(ece), float(max_gap)


def select_severity_features(X_train: pd.DataFrame, y_train: pd.Series, bootstrap_rounds: int):
    y_num = y_train.astype(float).reset_index(drop=True)
    X_part = X_train.reset_index(drop=True)
    rng = np.random.default_rng(RANDOM_STATE)
    rows = []

    for col in X_part.columns:
        corr = pd.Series(X_part[col]).corr(y_num, method="spearman")
        corr = 0.0 if pd.isna(corr) else float(corr)

        boot_corrs = []
        n = len(X_part)

        for _ in range(bootstrap_rounds):
            idx = rng.integers(0, n, size=n)
            c = pd.Series(X_part.iloc[idx][col]).corr(
                pd.Series(y_num.iloc[idx]),
                method="spearman",
            )
            c = 0.0 if pd.isna(c) else float(c)
            boot_corrs.append(c)

        rows.append({
            "feature": col,
            "spearman_with_fail": corr,
            "bootstrap_mean_abs_corr": float(np.mean(np.abs(boot_corrs))),
            "sign_agreement": float(np.mean(np.sign(boot_corrs) == np.sign(corr))) if corr != 0 else 1.0,
        })

    feature_table = pd.DataFrame(rows).sort_values(
        ["bootstrap_mean_abs_corr", "sign_agreement", "feature"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    keep = feature_table[
        (feature_table["bootstrap_mean_abs_corr"] >= 0.04) &
        (feature_table["sign_agreement"] >= 0.80)
    ].copy()

    if keep.empty:
        keep = feature_table.head(min(5, len(feature_table))).copy()

    keep = keep.sort_values(
        ["bootstrap_mean_abs_corr", "sign_agreement", "feature"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    keep["weight"] = keep["bootstrap_mean_abs_corr"] / keep["bootstrap_mean_abs_corr"].sum()

    selected = []
    corr_threshold = 0.95

    for _, row in keep.iterrows():
        feat = row["feature"]
        if not selected:
            selected.append(feat)
            continue

        max_abs_corr = max(
            abs(X_train[feat].corr(X_train[s], method="spearman"))
            for s in selected
        )
        if pd.isna(max_abs_corr) or max_abs_corr < corr_threshold:
            selected.append(feat)

    keep = keep[keep["feature"].isin(selected)].copy().reset_index(drop=True)
    keep["weight"] = keep["bootstrap_mean_abs_corr"] / keep["bootstrap_mean_abs_corr"].sum()

    return feature_table, keep


class SeverityScorer:
    def __init__(self, keep_table: pd.DataFrame, shrink_alpha: float = 1.0):
        self.keep_table = keep_table.copy()
        self.shrink_alpha = float(shrink_alpha)
        self.scaler = StandardScaler()
        self.edges = None
        self.raw_multiplier_map = None
        self.multiplier_map = None

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        keep_features = self.keep_table["feature"].tolist()
        self.scaler.fit(X_train[keep_features])

        train_scores = self.score(X_train)
        self.edges = np.quantile(train_scores, [1 / 3, 2 / 3])

        train_labels = self.labels_from_scores(train_scores)
        train_table = pd.DataFrame({"severity": train_labels, "fail": y_train.values})
        overall_fail_rate = float(y_train.mean())

        rates = train_table.groupby("severity")["fail"].mean().reindex(["Low", "Medium", "High"])
        self.raw_multiplier_map = (rates / overall_fail_rate).to_dict()
        self.multiplier_map = {
            k: float(1.0 + self.shrink_alpha * (v - 1.0))
            for k, v in self.raw_multiplier_map.items()
        }
        return self

    def score(self, X_raw: pd.DataFrame) -> np.ndarray:
        keep_features = self.keep_table["feature"].tolist()
        X_scaled = pd.DataFrame(
            self.scaler.transform(X_raw[keep_features]),
            columns=keep_features,
            index=X_raw.index,
        )

        score = np.zeros(len(X_scaled), dtype=float)
        for _, row in self.keep_table.iterrows():
            sign = 1.0 if row["spearman_with_fail"] >= 0 else -1.0
            score += X_scaled[row["feature"]].to_numpy(dtype=float) * float(row["weight"]) * sign
        return score

    def labels_from_scores(self, scores: np.ndarray) -> pd.Series:
        bins = np.digitize(scores, bins=self.edges, right=True)
        return pd.Series(np.array(["Low", "Medium", "High"])[bins])

    def labels(self, X_raw: pd.DataFrame) -> pd.Series:
        return self.labels_from_scores(self.score(X_raw))

    def value_table(self, severity_labels: pd.Series) -> pd.DataFrame:
        sev_mult = pd.Series(severity_labels).map(self.multiplier_map).astype(float).values
        row_missed_loss = BASE_MISSED_LOSS * sev_mult
        row_saved_value = row_missed_loss * RECOVERY_RATE
        row_check_cost = np.full(len(severity_labels), ROW_CHECK_COST, dtype=float)
        return pd.DataFrame({
            "row_check_cost": row_check_cost,
            "row_missed_loss": row_missed_loss,
            "row_saved_value": row_saved_value,
        })


def make_class_weight(style: str, y_train: pd.Series):
    if style == "none":
        return None

    pos_weight = (len(y_train) - int(y_train.sum())) / max(int(y_train.sum()), 1)

    if style == "balanced_x0.75":
        return {0: 1.0, 1: float(0.75 * pos_weight)}
    if style == "balanced":
        return {0: 1.0, 1: float(pos_weight)}
    if style == "balanced_x1.25":
        return {0: 1.0, 1: float(1.25 * pos_weight)}
    if style == "balanced_x1.5":
        return {0: 1.0, 1: float(1.5 * pos_weight)}

    raise ValueError(f"Unknown class weight style: {style}")


def build_base_estimator(model_name: str, class_weight):
    if model_name == "Logistic Regression":
        return Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000,
                solver="lbfgs",
                random_state=RANDOM_STATE,
                class_weight=class_weight,
            )),
        ])

    if model_name == "Random Forest":
        return Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=250,
                min_samples_leaf=2,
                class_weight=class_weight,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

    if model_name == "Extra Trees":
        return Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=350,
                min_samples_leaf=2,
                class_weight=class_weight,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

    if model_name == "SVM RBF":
        return Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                class_weight=class_weight,
                random_state=RANDOM_STATE,
            )),
        ])

    raise ValueError(f"Unknown model name: {model_name}")


def cv_model_search(X_train: pd.DataFrame, y_train: pd.Series) -> pd.DataFrame:
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []

    for model_name in ["Logistic Regression", "Random Forest", "Extra Trees", "SVM RBF"]:
        for weight_style in WEIGHT_STYLES:
            class_weight = make_class_weight(weight_style, y_train)

            fold_roc = []
            fold_pr = []

            for fit_idx, score_idx in cv.split(X_train, y_train):
                X_fit = X_train.iloc[fit_idx]
                y_fit = y_train.iloc[fit_idx]
                X_score = X_train.iloc[score_idx]
                y_score = y_train.iloc[score_idx]

                model = build_base_estimator(model_name, class_weight)
                model.fit(X_fit, y_fit)
                prob = clip_probs(model.predict_proba(X_score)[:, 1])

                fold_roc.append(roc_auc_score(y_score, prob))
                fold_pr.append(average_precision_score(y_score, prob))

            rows.append({
                "model": model_name,
                "weight_style": weight_style,
                "cv_pr_auc_mean": float(np.mean(fold_pr)),
                "cv_pr_auc_std": float(np.std(fold_pr)),
                "cv_roc_auc_mean": float(np.mean(fold_roc)),
                "cv_roc_auc_std": float(np.std(fold_roc)),
            })

    return pd.DataFrame(rows).sort_values(
        ["cv_pr_auc_mean", "cv_roc_auc_mean", "cv_pr_auc_std", "cv_roc_auc_std"],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)


def build_oof_raw_scores(X_train, y_train, model_name, weight_style):
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X_train), dtype=float)

    for fit_idx, hold_idx in cv.split(X_train, y_train):
        X_fit = X_train.iloc[fit_idx]
        y_fit = y_train.iloc[fit_idx]
        X_hold = X_train.iloc[hold_idx]

        class_weight = make_class_weight(weight_style, y_fit)
        model = build_base_estimator(model_name, class_weight)
        model.fit(X_fit, y_fit)
        oof[hold_idx] = clip_probs(model.predict_proba(X_hold)[:, 1])

    return oof


def fit_temperature_scaler(raw_scores, y_true):
    p = clip_probs(raw_scores)
    logits = np.log(p / (1.0 - p))
    y = np.asarray(y_true, dtype=float)

    def objective(temp):
        temp = max(float(temp), 1e-3)
        scaled = 1.0 / (1.0 + np.exp(-logits / temp))
        scaled = clip_probs(scaled)
        return float(-np.mean(y * np.log(scaled) + (1.0 - y) * np.log(1.0 - scaled)))

    res = minimize_scalar(objective, bounds=(0.05, 10.0), method="bounded")
    return float(res.x)


def apply_temperature_scaler(temp, raw_scores):
    p = clip_probs(raw_scores)
    logits = np.log(p / (1.0 - p))
    scaled = 1.0 / (1.0 + np.exp(-logits / max(float(temp), 1e-3)))
    return clip_probs(scaled)


def fit_histogram_binning(raw_scores, y_true, n_bins=5):
    p = clip_probs(raw_scores)
    y = np.asarray(y_true, dtype=float)

    quantiles = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(p, quantiles)
    edges[0] = 0.0
    edges[-1] = 1.0

    for i in range(1, len(edges)):
        if edges[i] <= edges[i - 1]:
            edges[i] = min(edges[i - 1] + 1e-8, 1.0)

    bin_ids = np.digitize(p, edges[1:-1], right=True)
    bin_rates = np.zeros(n_bins, dtype=float)
    global_rate = float(y.mean())

    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            bin_rates[b] = global_rate
        else:
            bin_rates[b] = float(y[mask].mean())

    return {"edges": edges, "bin_rates": bin_rates}


def apply_histogram_binning(calibrator, raw_scores):
    p = clip_probs(raw_scores)
    edges = calibrator["edges"]
    rates = calibrator["bin_rates"]
    bin_ids = np.digitize(p, edges[1:-1], right=True)
    out = np.array([rates[b] for b in bin_ids], dtype=float)
    return clip_probs(out)


def fit_venn_abers_model(model_name, weight_style, X_train, y_train):
    class_weight = make_class_weight(weight_style, y_train)
    estimator = build_base_estimator(model_name, class_weight)
    va = VennAbersCalibrator(
        estimator=estimator,
        inductive=True,
        cal_size=0.2,
        random_state=RANDOM_STATE,
    )
    va.fit(X_train, y_train)
    return va


def choose_calibration(X_train, y_train, X_validation, y_validation, model_name, weight_style):
    oof_raw = build_oof_raw_scores(
        X_train.reset_index(drop=True),
        y_train.reset_index(drop=True),
        model_name,
        weight_style,
    )

    class_weight = make_class_weight(weight_style, y_train)
    full_model = build_base_estimator(model_name, class_weight)
    full_model.fit(X_train, y_train)
    validation_raw = clip_probs(full_model.predict_proba(X_validation)[:, 1])

    candidates = {
        "none": validation_raw,
        "temperature": apply_temperature_scaler(
            fit_temperature_scaler(oof_raw, y_train.reset_index(drop=True).values),
            validation_raw
        ),
        "histogram_5bin": apply_histogram_binning(
            fit_histogram_binning(oof_raw, y_train.reset_index(drop=True).values, n_bins=5),
            validation_raw
        ),
    }

    if HAS_VENN_ABERS:
        venn_abers_model = fit_venn_abers_model(
            model_name,
            weight_style,
            X_train.reset_index(drop=True),
            y_train.reset_index(drop=True),
        )
        candidates["venn_abers"] = clip_probs(venn_abers_model.predict_proba(X_validation)[:, 1])

    rows = []
    for method, prob in candidates.items():
        _, ece, max_gap = calibration_stats(y_validation, prob)
        rows.append({
            "calibration": method,
            "validation_brier": float(brier_score_loss(y_validation, prob)),
            "validation_ece": float(ece),
            "validation_max_gap": float(max_gap),
        })

    summary = pd.DataFrame(rows).sort_values(
        ["validation_brier", "validation_ece", "validation_max_gap"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    winning_calibration = str(summary.iloc[0]["calibration"])
    return summary, winning_calibration, full_model, candidates[winning_calibration]


def build_threshold_table(y_true: pd.Series, scores: np.ndarray, value_table: pd.DataFrame) -> pd.DataFrame:
    table = value_table.copy().reset_index(drop=True)
    table["actual_failure"] = np.asarray(y_true, dtype=int)
    table["score"] = np.asarray(scores, dtype=float)
    table = table.sort_values("score", ascending=False).reset_index(drop=True)

    total_failures = max(int(table["actual_failure"].sum()), 1)
    rows = []

    for k in range(1, len(table) + 1):
        urgent = table.iloc[:k]
        backlog = table.iloc[k:]

        saved_failure_value = float(urgent.loc[urgent["actual_failure"] == 1, "row_saved_value"].sum())
        check_cost = float(urgent["row_check_cost"].sum())
        missed_failure_cost = float(backlog.loc[backlog["actual_failure"] == 1, "row_missed_loss"].sum())
        net_value = saved_failure_value - check_cost - missed_failure_cost

        tp = int(urgent["actual_failure"].sum())

        rows.append({
            "threshold": float(table.loc[k - 1, "score"]),
            "urgent_total": int(k),
            "net_value": net_value,
            "precision": tp / max(k, 1),
            "recall": tp / total_failures,
        })

    return pd.DataFrame(rows)


def build_capacity_table(y_true: pd.Series, scores: np.ndarray, value_table: pd.DataFrame, threshold: float):
    queue = value_table.copy().reset_index(drop=True)
    queue["actual_failure"] = np.asarray(y_true, dtype=int)
    queue["score"] = np.asarray(scores, dtype=float)
    queue = queue[queue["score"] >= threshold].sort_values("score", ascending=False).reset_index(drop=True)

    total_urgent_failures = max(int(queue["actual_failure"].sum()), 1)
    rows = []

    for k in range(1, len(queue) + 1):
        do_today = queue.iloc[:k]
        backlog = queue.iloc[k:]

        saved_failure_value = float(do_today.loc[do_today["actual_failure"] == 1, "row_saved_value"].sum())
        check_cost = float(do_today["row_check_cost"].sum())
        missed_failure_cost = float(backlog.loc[backlog["actual_failure"] == 1, "row_missed_loss"].sum())
        net_value = saved_failure_value - check_cost - missed_failure_cost

        rows.append({
            "capacity": int(k),
            "net_value": net_value,
            "today_precision": float(do_today["actual_failure"].mean()),
            "today_recall_within_urgent_pool": float(do_today["actual_failure"].sum() / total_urgent_failures),
        })

    return queue, pd.DataFrame(rows)


def fit_final_calibrator(X_train_final, y_train_final, model_name, weight_style, calibration_name):
    if calibration_name == "none":
        return None

    if calibration_name == "temperature":
        oof_raw = build_oof_raw_scores(
            X_train_final.reset_index(drop=True),
            y_train_final.reset_index(drop=True),
            model_name,
            weight_style,
        )
        return fit_temperature_scaler(oof_raw, y_train_final.reset_index(drop=True).values)

    if calibration_name == "histogram_5bin":
        oof_raw = build_oof_raw_scores(
            X_train_final.reset_index(drop=True),
            y_train_final.reset_index(drop=True),
            model_name,
            weight_style,
        )
        return fit_histogram_binning(oof_raw, y_train_final.reset_index(drop=True).values, n_bins=5)

    if calibration_name == "venn_abers":
        return fit_venn_abers_model(
            model_name,
            weight_style,
            X_train_final.reset_index(drop=True),
            y_train_final.reset_index(drop=True),
        )

    raise ValueError(f"Unknown calibration: {calibration_name}")


def apply_final_calibration(calibration_name, calibrator, raw_scores, X_test=None):
    if calibration_name == "none":
        return clip_probs(raw_scores)
    if calibration_name == "temperature":
        return apply_temperature_scaler(calibrator, raw_scores)
    if calibration_name == "histogram_5bin":
        return apply_histogram_binning(calibrator, raw_scores)
    if calibration_name == "venn_abers":
        return clip_probs(calibrator.predict_proba(X_test)[:, 1])

    raise ValueError(f"Unknown calibration: {calibration_name}")


def main():
    data = pd.read_csv(DATA_PATH).drop_duplicates().reset_index(drop=True)

    if TARGET not in data.columns:
        raise ValueError("The dataset must contain a 'fail' column.")

    feature_cols = [c for c in data.columns if c != TARGET and pd.api.types.is_numeric_dtype(data[c])]
    X = data[feature_cols].copy()
    y = data[TARGET].astype(int).copy()

    print_header("DATA")
    print(f"Data path: {DATA_PATH}")
    print(f"Rows: {len(data):,}")
    print(f"Failure rate: {y.mean():.1%}")
    print(f"Feature columns: {feature_cols}")

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    X_train, X_validation, y_train, y_validation = train_test_split(
        X_temp, y_temp,
        test_size=VALIDATION_SHARE_OF_REMAINDER,
        stratify=y_temp,
        random_state=RANDOM_STATE,
    )

    print_header("TRAIN ONLY SEVERITY")
    severity_feature_table, severity_keep = select_severity_features(X_train, y_train, BOOTSTRAP_ROUNDS)
    severity_scorer = SeverityScorer(severity_keep, shrink_alpha=SEVERITY_SHRINK_ALPHA).fit(X_train, y_train)

    print(severity_feature_table.to_string(index=False))
    print("\nKept severity features:")
    print(severity_keep[["feature", "weight"]].to_string(index=False))

    severity_multipliers = pd.DataFrame({
        "severity": ["Low", "Medium", "High"],
        "raw_multiplier": [severity_scorer.raw_multiplier_map[s] for s in ["Low", "Medium", "High"]],
        "shrunk_multiplier": [severity_scorer.multiplier_map[s] for s in ["Low", "Medium", "High"]],
    })
    print("\nSeverity multipliers:")
    print(severity_multipliers.to_string(index=False))

    print_header("TRAIN ONLY MODEL SEARCH")
    search_results = cv_model_search(X_train.reset_index(drop=True), y_train.reset_index(drop=True))
    print(search_results.to_string(index=False))

    winning_candidate = search_results.iloc[0]
    winning_model_name = str(winning_candidate["model"])
    winning_weight_style = str(winning_candidate["weight_style"])
    print(f"\nSelected candidate: {winning_model_name} with {winning_weight_style}")

    print_header("VALIDATION CALIBRATION CHOICE")
    calibration_summary, winning_calibration, _, validation_prob = choose_calibration(
        X_train, y_train, X_validation, y_validation, winning_model_name, winning_weight_style
    )
    print(calibration_summary.to_string(index=False))

    validation_values = severity_scorer.value_table(severity_scorer.labels(X_validation))

    print_header("VALIDATION THRESHOLD CHOICE")
    threshold_table = build_threshold_table(y_validation, validation_prob, validation_values)
    best_threshold_row = threshold_table.sort_values(
        ["net_value", "recall", "precision"],
        ascending=[False, False, False],
    ).iloc[0]

    locked_threshold = float(best_threshold_row["threshold"])
    print(best_threshold_row.to_string())

    print_header("VALIDATION CAPACITY CHOICE")
    validation_queue, validation_capacity_table = build_capacity_table(
        y_validation, validation_prob, validation_values, locked_threshold
    )

    max_capacity_value = float(validation_capacity_table["net_value"].max())
    retained = validation_capacity_table[
        validation_capacity_table["net_value"] >= CAPACITY_VALUE_RETENTION * max_capacity_value
    ].copy()

    selected_capacity = int(retained["capacity"].min())
    selected_capacity_row = validation_capacity_table.loc[
        validation_capacity_table["capacity"] == selected_capacity
    ].iloc[0]
    locked_capacity_share = selected_capacity / max(len(validation_queue), 1)

    print(selected_capacity_row.to_string())
    print(f"\nLocked capacity share: {locked_capacity_share:.6f}")

    print_header("LOCKED TEST")
    X_train_final = pd.concat([X_train, X_validation], axis=0).reset_index(drop=True)
    y_train_final = pd.concat([y_train, y_validation], axis=0).reset_index(drop=True)

    _, final_keep = select_severity_features(X_train_final, y_train_final, BOOTSTRAP_ROUNDS)
    final_scorer = SeverityScorer(final_keep, shrink_alpha=SEVERITY_SHRINK_ALPHA).fit(X_train_final, y_train_final)
    test_values = final_scorer.value_table(final_scorer.labels(X_test))

    winning_class_weight = make_class_weight(winning_weight_style, y_train_final)
    final_model = build_base_estimator(winning_model_name, winning_class_weight)
    final_model.fit(X_train_final, y_train_final)

    test_raw = clip_probs(final_model.predict_proba(X_test)[:, 1])
    final_calibrator = fit_final_calibrator(
        X_train_final, y_train_final, winning_model_name, winning_weight_style, winning_calibration
    )
    test_prob = apply_final_calibration(
        winning_calibration, final_calibrator, test_raw, X_test=X_test
    )

    _, test_ece, test_max_gap = calibration_stats(y_test, test_prob)
    test_queue, test_capacity_table = build_capacity_table(y_test, test_prob, test_values, locked_threshold)

    locked_test_capacity = max(1, int(round(locked_capacity_share * len(test_queue))))
    locked_test_capacity = min(locked_test_capacity, len(test_queue))
    locked_test_row = test_capacity_table.loc[
        test_capacity_table["capacity"] == locked_test_capacity
    ].iloc[0]

    test_summary = pd.DataFrame([{
        "model": winning_model_name,
        "weight_style": winning_weight_style,
        "calibration": winning_calibration,
        "shrink_alpha": SEVERITY_SHRINK_ALPHA,
        "threshold": locked_threshold,
        "capacity_share": locked_capacity_share,
        "test_urgent_total": int(len(test_queue)),
        "test_capacity": int(locked_test_capacity),
        "test_roc_auc": float(roc_auc_score(y_test, test_prob)),
        "test_pr_auc": float(average_precision_score(y_test, test_prob)),
        "test_brier": float(brier_score_loss(y_test, test_prob)),
        "test_ece": float(test_ece),
        "test_max_gap": float(test_max_gap),
        "test_capacity_net_value": float(locked_test_row["net_value"]),
        "test_today_precision": float(locked_test_row["today_precision"]),
        "test_today_recall_within_urgent_pool": float(locked_test_row["today_recall_within_urgent_pool"]),
    }])

    row = test_summary.iloc[0]
    for k, v in row.items():
        print(f"{k:35} {v}")

    output = X_test.copy().reset_index(drop=True)
    output.insert(0, "machine_id", np.arange(1, len(output) + 1))
    output["actual_fail"] = y_test.reset_index(drop=True).values
    output["predicted_failure_probability"] = test_prob
    output["severity_band"] = final_scorer.labels(X_test).values
    output["is_urgent"] = output["predicted_failure_probability"] >= locked_threshold
    output = output.sort_values("predicted_failure_probability", ascending=False).reset_index(drop=True)
    output["queue_rank"] = np.arange(1, len(output) + 1)
    output["action"] = "Monitor"
    output.loc[
        (output["is_urgent"]) & (output["queue_rank"] <= locked_test_capacity),
        "action"
    ] = "Inspect now"
    output.loc[
        (output["is_urgent"]) & (output["queue_rank"] > locked_test_capacity),
        "action"
    ] = "Backlog urgent"

    output.to_csv("locked_test_queue.csv", index=False)
    test_summary.to_csv("locked_test_summary.csv", index=False)

    print("\nWrote: locked_test_queue.csv and locked_test_summary.csv")


if __name__ == "__main__":
    main()


DATA
Data path: /kaggle/input/datasets/umerrtx/machine-failure-prediction-using-sensor-data/data.csv
Rows: 943
Failure rate: 41.7%
Feature columns: ['footfall', 'tempMode', 'AQ', 'USS', 'CS', 'VOC', 'RP', 'IP', 'Temperature']

TRAIN ONLY SEVERITY
    feature  spearman_with_fail  bootstrap_mean_abs_corr  sign_agreement
        VOC            0.765306                 0.765613          1.0000
         AQ            0.593366                 0.592437          1.0000
        USS           -0.479078                 0.478772          1.0000
   footfall           -0.213508                 0.213855          1.0000
Temperature            0.208959                 0.210468          1.0000
         IP            0.099760                 0.102374          0.9875
         CS           -0.068232                 0.073826          0.9500
   tempMode           -0.039375                 0.041257          0.8375
         RP            0.001400                 0.029317          0.5375

Kept severity feature

# Predictive Maintenance Triage

This notebook ranks machines by failure risk and turns that ranking into an inspection queue.

It decides:

- which machines should be inspected now
- which can wait
- how much of the urgent queue can be handled today

The notebook ends up using:

- model: Extra Trees
- class weighting: `balanced_x1.5`
- calibration: `none`
- severity multipliers shrunk toward `1.0` with `shrink_alpha = 0.5`
- urgent cutoff: the validation threshold with the highest net value
- daily queue size: the smallest validation capacity that keeps at least 95 percent of the best capacity value

For each machine, the notebook outputs:

- predicted failure probability
- severity band
- queue rank
- action:
  - `Inspect now`
  - `Backlog urgent`
  - `Monitor`

# Data split

The data is split into train, validation, and test.

- train fits the model
- validation sets the queue rule
- test is used once at the end

The test set is not used to choose the model or the queue rule.

# Severity

Not every likely failure is treated the same way.

The notebook builds a severity score from the training set.

For each numeric feature, it checks:

- Spearman correlation with failure
- bootstrap mean absolute correlation
- bootstrap sign agreement

What these mean:

- Spearman correlation checks whether the feature tends to rise or fall with failure
- bootstrap mean absolute correlation checks whether that link stays strong across many resampled training sets
- bootstrap sign agreement checks whether that link keeps the same direction

Example:

if higher `VOC` usually comes with more failures, that is a positive link  
if that same link stays strong and keeps the same direction across many resampled training sets, `VOC` is treated as more reliable

Only features with enough signal and enough directional stability are kept.

Kept severity features in this run:

- VOC
- AQ
- USS
- footfall
- Temperature
- IP
- CS
- tempMode

Feature weights come from bootstrap mean absolute correlation.

A feature with stronger and more stable failure signal gets more weight in the severity score.

In this run, the largest weights are on:

- VOC
- AQ
- USS

A stronger warning signal counts more.
A weaker or noisier signal counts less.

# Severity bands and multipliers

The severity score is split into three bands:

- Low
- Medium
- High

The cut points come from train score quantiles.

Then the notebook checks how often failure appears inside each band and compares that with the overall train failure rate. That gives one multiplier per band.

Raw multipliers in this run:

- Low: `0.054463`
- Medium: `0.711249`
- High: `2.232975`

These values are pulled toward `1.0` with:

`1 + shrink_alpha * (raw_multiplier - 1)`

using:

`shrink_alpha = 0.5`

Final shrunk multipliers:

- Low: `0.527231`
- Medium: `0.855625`
- High: `1.616488`

This keeps the band order but makes the gap between bands smaller.

These are not measured dollar losses.
They are relative consequence multipliers learned from train.

# Model search

The notebook compares four model families on train only cross validation:

- Logistic Regression
- Random Forest
- Extra Trees
- SVM with RBF kernel

It also tests a small class weight grid:

- `none`
- `balanced_x0.75`
- `balanced`
- `balanced_x1.25`
- `balanced_x1.5`

Winner:

- Extra Trees
- `balanced_x1.5`

So, in this search space, Extra Trees with `balanced_x1.5` did the best job of putting the failing machines above the non failing ones.

# Calibration

Calibration asks whether a predicted probability matches what really happens.

That is different from ranking.

Ranking is about order.

If apple A looks more rotten than apple B, apple A should be placed above apple B.  
If machine A looks more failure prone than machine B, machine A should be placed above machine B.

So ranking asks whether the list is in the right order.
Calibration asks whether the number itself is right.

Calibration methods kept in the notebook:

- `none`
- `temperature`
- `histogram_5bin`
- `venn_abers` when available

Final choice:

- `none`

Some methods improved one calibration metric but made others worse, so raw probabilities stayed the best overall tradeoff in this run.

The model does a better job ordering machines than making exact probability statements.
It is good at moving the more failure prone machines to the top of the list.

# Threshold

After validation probabilities are produced, the notebook sorts machines from highest score to lowest score.

Then it checks every possible cutoff.

At each cutoff, it computes:

- value saved by catching urgent failures
- cost of checking urgent machines
- cost of missed failures left in backlog

The cutoff with the highest validation net value is selected.

Chosen threshold:

- `0.047639`

So the urgent line is set by value, not by accuracy or F1.

# Daily capacity

After the urgent queue is built, the notebook still has to decide how far down that queue to act today.

It checks each possible queue depth and asks how much value is kept if work stops there.

Then it chooses the smallest capacity that keeps at least 95 percent of the best validation value.

Chosen validation result:

- capacity = `67`
- locked capacity share = `0.626168`

A capacity share is locked instead of a fixed count because the urgent pool can change in size.

Important limit:

the dataset does not contain staffing logs  
so capacity is still a policy rule, not something measured directly from the data

# Locked test result

After all choices are fixed, the notebook refits on train plus validation and runs once on test.

Locked test result:

- model: Extra Trees
- weight style: `balanced_x1.5`
- calibration: `none`
- threshold: `0.047639`
- capacity share: `0.626168`
- test urgent total: `117`
- test capacity: `73`
- ROC AUC: `0.978763`
- PR AUC: `0.965627`
- Brier: `0.062474`
- ECE: `0.100361`
- capacity net value: `41882.275449`
- today precision: `0.808219`
- today recall within urgent pool: `1.0`

# How the final queue behaves

The model ranks machines well.

Evidence:

- ROC AUC near `0.979`
- PR AUC near `0.966`

That means failing machines tend to rise to the top of the list.

The model does a better job ordering machines than making exact probability statements.
That is why calibration stayed at `none`.

The final policy gives:

- an urgent queue
- a daily cutoff inside that queue
- positive value on test


# Assumed value outside the dataset:

- `ROW_CHECK_COST = 10.0`
- `BASE_MISSED_LOSS = 500.0`
- `RECOVERY_RATE = 1.0`
- `row_saved_value = row_missed_loss * recovery_rate`
- the 95 percent value retention rule
- the three severity bands

`10` and `500` come from the APS Failure at Scania Trucks benchmark. `1.0` does not. It is a notebook assumption: missed loss is treated as avoidable loss, so a timely inspection is counted as recovering all of it. The literature supports the idea of recoverable value from timely action, but not one fixed recovery rate. References: [Scania APS benchmark][1] and [Siemens predictive maintenance material][2].

The dataset does not contain staffing logs, repair cost, downtime cost, or realized savings after inspection.

# Values learned from the data:

- severity features
- severity weights
- severity band cut points
- severity failure ratios
- winning model
- winning class weight
- winning threshold
- locked capacity share from the validation curve

[1]: https://archive.ics.uci.edu/dataset/421/aps%2Bfailure%2Bat%2Bscania%2Btrucks
[2]: https://assets.new.siemens.com/siemens/assets/api/uuid%3A1b43afb5-2d07-47f7-9eb7-893fe7d0bc59/TCOD-2024_original.pdf

Boundary : 
Operational capacity and business value parameters are not directly observable in the dataset and therefore remain explicit policy assumptions rather than data derived quantities.

Even shorter:

Calibration is improvable.
Capacity and dollar value assumptions are not discoverable here.

---
Calibration

Imagine three students guessing which apples are rotten.

Student A says:
this apple is worse than that apple,
and that apple is worse than the next one.

If Student A gets the order right, that helps you pick which apples to check first.

But calibration is harder.
Now you ask Student A:
how many chances out of 100 is this apple rotten?

Getting that exact number right is much harder than just getting the order right.

Your model is like that.
It is good at saying:
check this machine before that one.
It is less good at saying:
this machine is exactly 63 percent likely to fail.

The model is very good at ordering machines by risk, but only moderately good at making exact probability statements, which is common when tree models are trained on a relatively small dataset.

---
Weights

Class weights
magine you are telling the model how loudly to listen to failure examples.

none means normal volume
balanced means louder
balanced_x1.5 means even louder

etc : none
balanced_x0.75
balanced
balanced_x1.25
balanced_x1.5

severity feature weights

If a feature is much better at warning about failure,
it should count more in the severity score.

That means stronger features get bigger weight in the severity score.

For example, in your results:
VOC got the biggest weight,
then AQ,
then USS,
because they had the strongest stable signal.

model class weights handle imbalance,
severity feature weights handle how much each sensor matters in the severity score.